# 06｜Forensics Adapter 訓練資料準備

將 02 的人臉裁切結果整理成官方 Forensics Adapter 專案可讀取的 `frames/`、`masks/` 與 JSON 索引。

完整方法除了 Real／Fake 標籤，還需要每張圖片的 manipulation mask 作為 blending boundary 的監督訊號。這一步負責產生對齊的 mask。

關鍵約束：

- **不重新切分資料**，沿用 `face_manifest.csv` 既有的 train／val／test。
- Fake 的 mask 必須來自 FF++ 官方的 Deepfakes mask 影片，Real 的 mask 固定為全黑。
- mask 使用與人臉**完全相同的裁切座標**同步裁切，否則監督訊號會與影像錯位。
- 若 Fake mask 影片不完整，程式中止，不以全黑 mask 代替。全黑 mask 代表「這張圖沒有任何偽造區域」，用在 Fake 樣本上會直接給出錯誤的監督。

In [1]:
from pathlib import Path
import json
import math
import os
import platform
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


def setup_chinese_font():
    system = platform.system()
    if system == "Windows":
        candidates = ["Microsoft JhengHei", "Microsoft YaHei", "SimHei"]
    elif system == "Darwin":
        candidates = ["PingFang TC", "Heiti TC", "Arial Unicode MS"]
    else:
        candidates = [
            "Noto Sans CJK TC",
            "Noto Sans CJK SC",
            "WenQuanYi Zen Hei",
        ]
    plt.rcParams["font.sans-serif"] = candidates + ["DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False


setup_chinese_font()
print("OpenCV：", cv2.__version__)

OpenCV： 4.12.0


## 1. 設定

In [2]:
# 02 Notebook 產生的人臉 manifest
FACE_MANIFEST = Path(
    "./FFPP_Deepfakes_16frames_faces/metadata/face_manifest.csv"
)

# FaceForensics++ 原始資料根目錄
FFPP_ROOT = Path(r"D:\資料\專題\FF++")

# 本 Notebook 的獨立輸出資料夾
OUTPUT_ROOT = Path("./FFPP_ForensicsAdapter_Data")
DATASET_JSON_FOLDER = OUTPUT_ROOT / "dataset_json"

# 沿用目前 Baseline 的人臉輸入；官方模型載入時仍會轉成 256×256
OUTPUT_SIZE = 256

# 第一次只處理少量資料；正式執行時改成 None
MAX_ITEMS = None

# 安全開關
RUN_PREPARATION = True

# False 時保留已完成結果並續跑
OVERWRITE = False

print("人臉 manifest：", FACE_MANIFEST.resolve())
print("FF++ 根目錄：", FFPP_ROOT.resolve())
print("輸出資料夾：", OUTPUT_ROOT.resolve())
print("是否執行：", RUN_PREPARATION)

人臉 manifest： D:\資料\專題\PY\deepfake\新版專題研究\FFPP_Deepfakes_16frames_faces\metadata\face_manifest.csv
FF++ 根目錄： D:\資料\專題\FF++
輸出資料夾： D:\資料\專題\PY\deepfake\新版專題研究\FFPP_ForensicsAdapter_Data
是否執行： True


## 2. 讀取並檢查既有人臉 manifest

必須包含 `crop_x1`～`crop_y2` 四個裁切座標欄位。mask 的裁切完全依賴這些座標，而非重新偵測人臉——重新偵測會因隨機性導致框位些微偏移，讓 mask 與影像對不齊。

In [3]:
if not FACE_MANIFEST.is_file():
    raise FileNotFoundError(f"找不到 face_manifest.csv：{FACE_MANIFEST}")
if not FFPP_ROOT.is_dir():
    raise FileNotFoundError(f"找不到 FF++ 根目錄：{FFPP_ROOT}")

face_df = pd.read_csv(FACE_MANIFEST)
required_columns = {
    "face_path",
    "video_stem",
    "frame_index",
    "split",
    "label",
    "class_name",
    "crop_x1",
    "crop_y1",
    "crop_x2",
    "crop_y2",
}
missing_columns = sorted(required_columns - set(face_df.columns))
if missing_columns:
    raise ValueError(f"face_manifest.csv 缺少欄位：{missing_columns}")

face_df = face_df[
    face_df["split"].isin(["train", "val", "test"])
    & face_df["class_name"].isin(["real", "fake"])
].copy()
face_df["frame_index"] = face_df["frame_index"].astype(int)
face_df["label"] = face_df["label"].astype(int)

missing_faces = [
    path for path in face_df["face_path"].astype(str)
    if not Path(path).is_file()
]
if missing_faces:
    raise FileNotFoundError(
        f"manifest 中有 {len(missing_faces)} 張人臉圖片不存在，"
        f"範例：{missing_faces[0]}"
    )

if MAX_ITEMS is not None:
    selected_df = (
        face_df.sort_values(["split", "class_name", "video_stem", "frame_index"])
        .groupby(["split", "class_name"], group_keys=False)
        .head(max(1, MAX_ITEMS // 6))
        .reset_index(drop=True)
    )
else:
    selected_df = face_df.reset_index(drop=True)

display(pd.crosstab(
    [selected_df["split"], selected_df["class_name"]],
    columns="images",
))
print("本次預計處理：", len(selected_df))

col_0             images
split class_name        
test  fake          2240
      real          2240
train fake         11520
      real         11518
val   fake          2239
      real          2239

本次預計處理： 31996


## 3. 定位 FF++ 官方 mask 影片

FF++ 依下載方式不同，mask 可能位於數種路徑，程式逐一檢查。1,000 部 Fake 影片全數找到對應的 mask 影片。

In [4]:
def mask_video_candidates(video_stem):
    base = FFPP_ROOT / "manipulated_sequences" / "Deepfakes"
    return [
        base / "masks" / "videos" / f"{video_stem}.mp4",
        base / "masks" / f"{video_stem}.mp4",
        base / "c23" / "masks" / "videos" / f"{video_stem}.mp4",
        base / "c23" / "masks" / f"{video_stem}.mp4",
    ]


def resolve_mask_video(video_stem):
    for candidate in mask_video_candidates(video_stem):
        if candidate.is_file():
            return candidate
    return None


fake_videos = sorted(
    selected_df.loc[
        selected_df["class_name"].eq("fake"), "video_stem"
    ].unique()
)
mask_lookup = {
    video_stem: resolve_mask_video(video_stem)
    for video_stem in fake_videos
}
missing_mask_videos = [
    video_stem for video_stem, path in mask_lookup.items()
    if path is None
]

print("本次 Fake 影片數：", len(fake_videos))
print("找到 mask 影片：", len(fake_videos) - len(missing_mask_videos))
print("缺少 mask 影片：", len(missing_mask_videos))
if missing_mask_videos:
    print("缺少範例：", missing_mask_videos[:10])
    print(
        "請先從 FF++ 官方資料補下載 Deepfakes masks，"
        "再執行正式資料準備。"
    )

本次 Fake 影片數： 1000
找到 mask 影片： 1000
缺少 mask 影片： 0


## 4. 同步裁切影像與 mask

mask 與影像使用相同座標，但補邊與插值方式不同：

- 補邊：影像用 `BORDER_REFLECT_101`，mask 用 `BORDER_CONSTANT` 填 0。鏡射補出來的 mask 會在畫面外憑空造出偽造區域。
- 插值：影像用 `INTER_AREA`，mask 用 `INTER_NEAREST`。線性插值會在 mask 邊緣產生中間灰階，破壞二值語意。

縮放後再以閾值 10 重新二值化，去除殘留的中間值。

In [5]:
def crop_with_manifest(array, row, output_size, is_mask=False):
    height, width = array.shape[:2]
    x1 = int(row.crop_x1)
    y1 = int(row.crop_y1)
    x2 = int(row.crop_x2)
    y2 = int(row.crop_y2)

    pad_left = max(0, -x1)
    pad_top = max(0, -y1)
    pad_right = max(0, x2 - width)
    pad_bottom = max(0, y2 - height)

    clipped_x1 = max(0, x1)
    clipped_y1 = max(0, y1)
    clipped_x2 = min(width, x2)
    clipped_y2 = min(height, y2)
    crop = array[clipped_y1:clipped_y2, clipped_x1:clipped_x2]
    if crop.size == 0:
        return None

    if any([pad_left, pad_top, pad_right, pad_bottom]):
        border_type = cv2.BORDER_CONSTANT if is_mask else cv2.BORDER_REFLECT_101
        crop = cv2.copyMakeBorder(
            crop,
            pad_top,
            pad_bottom,
            pad_left,
            pad_right,
            borderType=border_type,
            value=0,
        )

    interpolation = cv2.INTER_NEAREST if is_mask else cv2.INTER_AREA
    return cv2.resize(
        crop,
        (output_size, output_size),
        interpolation=interpolation,
    )


def read_video_frame(video_path, frame_index):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
    ok, frame = cap.read()
    cap.release()
    return frame if ok else None


def output_paths(row):
    relative = (
        Path(row.split)
        / row.class_name
        / row.video_stem
        / f"frame_{int(row.frame_index):06d}.png"
    )
    return OUTPUT_ROOT / "frames" / relative, OUTPUT_ROOT / "masks" / relative


def prepare_one(row):
    frame_path, mask_path = output_paths(row)
    if (
        not OVERWRITE
        and frame_path.is_file()
        and mask_path.is_file()
    ):
        return {
            "status": "ok",
            "frame_path": str(frame_path.resolve()),
            "mask_path": str(mask_path.resolve()),
        }

    face = cv2.imread(str(row.face_path))
    if face is None:
        return {"status": "face_read_error"}
    face = cv2.resize(
        face,
        (OUTPUT_SIZE, OUTPUT_SIZE),
        interpolation=cv2.INTER_AREA,
    )

    if row.class_name == "real":
        mask_crop = np.zeros(
            (OUTPUT_SIZE, OUTPUT_SIZE),
            dtype=np.uint8,
        )
    else:
        mask_video = mask_lookup.get(row.video_stem)
        if mask_video is None:
            return {"status": "mask_video_missing"}
        mask_frame = read_video_frame(mask_video, row.frame_index)
        if mask_frame is None:
            return {"status": "mask_frame_read_error"}
        mask_gray = cv2.cvtColor(mask_frame, cv2.COLOR_BGR2GRAY)
        mask_crop = crop_with_manifest(
            mask_gray,
            row,
            OUTPUT_SIZE,
            is_mask=True,
        )
        if mask_crop is None:
            return {"status": "mask_crop_error"}
        _, mask_crop = cv2.threshold(
            mask_crop,
            10,
            255,
            cv2.THRESH_BINARY,
        )

    frame_path.parent.mkdir(parents=True, exist_ok=True)
    mask_path.parent.mkdir(parents=True, exist_ok=True)
    frame_ok = cv2.imwrite(str(frame_path), face)
    mask_ok = cv2.imwrite(str(mask_path), mask_crop)
    if not frame_ok or not mask_ok:
        return {"status": "write_error"}

    return {
        "status": "ok",
        "frame_path": str(frame_path.resolve()),
        "mask_path": str(mask_path.resolve()),
    }

In [6]:
if not RUN_PREPARATION:
    print("RUN_PREPARATION=False，目前只完成檢查，尚未寫入資料。")
    prepared_df = pd.DataFrame()
elif missing_mask_videos:
    raise FileNotFoundError(
        "仍有 Fake mask 影片缺失，為避免錯誤監督，已停止正式處理。"
    )
else:
    records = []
    start = time.time()
    for row in tqdm(
        selected_df.itertuples(index=False),
        total=len(selected_df),
        desc="建立 Forensics Adapter 資料",
    ):
        result = prepare_one(row)
        records.append({
            "video_stem": row.video_stem,
            "frame_index": int(row.frame_index),
            "split": row.split,
            "class_name": row.class_name,
            "label": int(row.label),
            **result,
        })

    prepared_df = pd.DataFrame(records)
    metadata_dir = OUTPUT_ROOT / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = metadata_dir / "forensics_adapter_manifest.csv"
    prepared_df.to_csv(
        manifest_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"完成，耗時 {(time.time() - start) / 60:.1f} 分鐘")
    print("manifest：", manifest_path)
    display(prepared_df["status"].value_counts(dropna=False))

建立 Forensics Adapter 資料:   0%|          | 0/31996 [00:00<?, ?it/s]

完成，耗時 21.0 分鐘
manifest： FFPP_ForensicsAdapter_Data\metadata\forensics_adapter_manifest.csv


status
ok    31996
Name: count, dtype: int64

## 5. 建立 train／val／test JSON

轉換成官方 Dataset 的巢狀格式，Real 對應 `FF-real`、Fake 對應 `FF-DF`。若有任何一筆資料處理失敗，不建立 JSON——部分缺漏的資料集會讓訓練在中途才報錯，比一開始就中止更難排查。

In [7]:
DATASET_NAMES = {
    "train": "FFPP-DF-Local-Train",
    "val": "FFPP-DF-Local-Val",
    "test": "FFPP-DF-Local-Test",
}


def build_dataset_json(ok_df, split_name):
    dataset_name = DATASET_NAMES[split_name]
    mode_name = "train" if split_name == "train" else "test"
    content = {
        dataset_name: {
            "FF-real": {"train": {}, "test": {}},
            "FF-DF": {"train": {}, "test": {}},
        }
    }

    subset = ok_df[ok_df["split"].eq(split_name)].copy()
    for (class_name, video_stem), group in subset.groupby(
        ["class_name", "video_stem"]
    ):
        label_name = "FF-real" if class_name == "real" else "FF-DF"
        frames = (
            group.sort_values("frame_index")["frame_path"]
            .astype(str)
            .tolist()
        )
        content[dataset_name][label_name][mode_name][video_stem] = {
            "label": label_name,
            "frames": frames,
        }
    return content


if prepared_df.empty:
    print("尚未有正式處理結果，因此不建立 JSON。")
else:
    failed = prepared_df[~prepared_df["status"].eq("ok")]
    if len(failed):
        display(failed.head(20))
        raise RuntimeError(
            f"仍有 {len(failed)} 筆失敗資料。請先排除後再建立 JSON。"
        )

    DATASET_JSON_FOLDER.mkdir(parents=True, exist_ok=True)
    json_paths = {}
    for split_name, dataset_name in DATASET_NAMES.items():
        content = build_dataset_json(prepared_df, split_name)
        path = DATASET_JSON_FOLDER / f"{dataset_name}.json"
        path.write_text(
            json.dumps(content, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        json_paths[split_name] = path
        print(f"{split_name}：{path}")

train：FFPP_ForensicsAdapter_Data\dataset_json\FFPP-DF-Local-Train.json
val：FFPP_ForensicsAdapter_Data\dataset_json\FFPP-DF-Local-Val.json
test：FFPP_ForensicsAdapter_Data\dataset_json\FFPP-DF-Local-Test.json


## 6. 品質檢查

驗證三個 split 之間沒有重複的 `video_stem`，確認切分在轉換過程中沒有被破壞。

31,996 張圖片全數處理成功，無失敗案例。

原本此處為人臉、mask 與疊合結果的三欄對照圖，用於目視確認 mask 與臉部區域對齊，因 FaceForensics++ 授權條款限制影像再散布，已於公開版本移除。程式碼保留，可在本機重現。

In [9]:
if prepared_df.empty:
    print("尚無正式結果可檢查。")
else:
    ok_df = prepared_df[prepared_df["status"].eq("ok")].copy()
    summary = (
        ok_df.groupby(["split", "class_name"])
        .agg(
            images=("frame_path", "size"),
            videos=("video_stem", "nunique"),
        )
        .reset_index()
    )
    display(summary)

    overlap = {}
    split_videos = {
        split: set(ok_df.loc[ok_df["split"].eq(split), "video_stem"])
        for split in ["train", "val", "test"]
    }
    overlap["train_val"] = len(
        split_videos["train"] & split_videos["val"]
    )
    overlap["train_test"] = len(
        split_videos["train"] & split_videos["test"]
    )
    overlap["val_test"] = len(
        split_videos["val"] & split_videos["test"]
    )
    print("video_stem 重疊檢查：", overlap)
    if any(overlap.values()):
        raise ValueError("不同 split 出現相同 video_stem。")

    fake_examples = ok_df[ok_df["class_name"].eq("fake")].head(4)
    fig, axes = plt.subplots(
        len(fake_examples),
        3,
        figsize=(9, 3 * len(fake_examples)),
    )
    axes = np.atleast_2d(axes)
    for row_axes, (_, row) in zip(axes, fake_examples.iterrows()):
        def imread_unicode(path, flags=cv2.IMREAD_COLOR):
            data = np.fromfile(str(path), dtype=np.uint8)
            return cv2.imdecode(data, flags)
        
        
        image_bgr = imread_unicode(
            row["frame_path"],
            cv2.IMREAD_COLOR,
        )
        mask = imread_unicode(
            row["mask_path"],
            cv2.IMREAD_GRAYSCALE,
        )
        
        if image_bgr is None:
            raise FileNotFoundError(
                f"無法讀取圖片：{row['frame_path']}"
            )
        
        if mask is None:
            raise FileNotFoundError(
                f"無法讀取 Mask：{row['mask_path']}"
            )
        
        image = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )
        overlay = image.copy()
        overlay[mask > 0] = (
            0.55 * overlay[mask > 0]
            + 0.45 * np.array([255, 60, 60])
        ).astype(np.uint8)

        row_axes[0].imshow(image)
        row_axes[0].set_title("人臉")
        row_axes[1].imshow(mask, cmap="gray")
        row_axes[1].set_title("Manipulation mask")
        row_axes[2].imshow(overlay)
        row_axes[2].set_title("疊合檢查")
        for axis in row_axes:
            axis.axis("off")
    plt.tight_layout()
    plt.show()

    print("資料準備完成。下一步執行 07 Notebook。")

,split,class_name,images,videos
0,test,fake,2240,140
1,test,real,2240,140
2,train,fake,11520,720
3,train,real,11518,720
4,val,fake,2239,140
5,val,real,2239,140


video_stem 重疊檢查： {'train_val': 0, 'train_test': 0, 'val_test': 0}


資料準備完成。下一步執行 07 Notebook。
